## PCAM + GRAPH + PAGERANK ATTENTION

In [19]:
import os
import h5py
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import timm
import networkx as nx

from tqdm import tqdm
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from torch_geometric.data import Data
from torch_geometric.nn import GATConv

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


# =========================================
# 1. LOAD PCAM DATA
# =========================================

def load_pcam(path, n_samples=5000):
    with h5py.File(os.path.join(path, 'camelyonpatch_level_2_split_train_x.h5'), 'r') as f:
        X = f['x'][:n_samples]

    with h5py.File(os.path.join(path, 'camelyonpatch_level_2_split_train_y.h5'), 'r') as f:
        y = f['y'][:n_samples]

    return X, y.reshape(-1)


# =========================================
# 2. FEATURE EXTRACTION (ViT)
# =========================================

def extract_features(X, device):
    X = torch.tensor(X / 255.0).permute(0,3,1,2).float()

    vit = timm.create_model('vit_base_patch16_224', pretrained=True)
    vit.head = nn.Identity()
    vit = vit.to(device)
    vit.eval()

    def resize(x):
        return F.interpolate(x, size=(224,224))

    features = []

    with torch.no_grad():
        for i in tqdm(range(0, len(X), 32)):
            batch = resize(X[i:i+32].to(device))
            feat = vit.forward_features(batch)
            feat = feat.mean(dim=1)
            features.append(feat.cpu())

    return torch.cat(features).numpy()


# =========================================
# 3. GRAPH CONSTRUCTION
# =========================================

def build_graph(features, k=10):
    A = kneighbors_graph(features, k, mode='distance')

    sigma = np.mean(A.data)
    A.data = np.exp(-(A.data**2) / (sigma**2))

    A = A.minimum(A.T)  # mutual graph
    return A


# =========================================
# 4. PAGERANK COMPUTATION
# =========================================

def compute_pagerank(A):
    G = nx.from_scipy_sparse_array(A)
    pr = nx.pagerank(G, alpha=0.85)

    pr_scores = np.array([pr[i] for i in range(len(pr))])
    return pr_scores


# =========================================
# 5. DATA PREPARATION
# =========================================

def prepare_data(X_feat, pr_scores, A, y):

    # Feature augmentation
    X_aug = np.concatenate([X_feat, pr_scores[:, None]], axis=1)

    scaler = StandardScaler()
    X_aug = scaler.fit_transform(X_aug)

    edge_index = np.vstack(A.nonzero())

    data = Data(
        x=torch.tensor(X_aug).float(),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        y=torch.tensor(y).long()
    )

    return data


# =========================================
# 6. PAGERANK-AWARE GAT
# =========================================

class PageRankGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()

        self.gat = GATConv(in_dim, hidden_dim, heads=4)
        self.fc = nn.Linear(hidden_dim * 4, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index, pagerank):
        x = self.gat(x, edge_index)
        x = self.dropout(x)
    
        # normalize pagerank
        pr = pagerank / pagerank.max()

        # attention-style scaling (soft)
        x = x * (1 + pr.unsqueeze(1))

        return self.fc(x)


In [20]:
# =========================================
# 7. TRAIN FUNCTION
# =========================================
def train(model, data, optimizer, criterion, train_mask):

    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.pr)

    loss = criterion(out[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()

    return loss.item()


# =========================================
# 8. EVALUATION
# =========================================
def evaluate(model, data, mask):

    model.eval()

    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.pr)
        probs = torch.softmax(logits, dim=1)[:,1]
        preds = logits.argmax(dim=1)

    y_true = data.y[mask].cpu().numpy()
    y_pred = preds[mask].cpu().numpy()
    y_prob = probs[mask].cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    return acc, f1, auc


In [21]:
# =========================================
# 9. MAIN PIPELINE
# =========================================

def main():

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    print("Loading data...")
    X, y = load_pcam('../data', 5000)

    print("Extracting features...")
    X_feat = extract_features(X, device)

    print("Building graph...")
    A = build_graph(X_feat)

    print("Computing PageRank...")
    pr_scores = compute_pagerank(A)

    print("Preparing data...")
    data = prepare_data(X_feat, pr_scores, A, y)

    # Attach pagerank tensor
    data.pr = torch.tensor(pr_scores).float()

    # Train/test split
    idx = np.arange(len(y))
    train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=y)

    train_mask = torch.zeros(len(y), dtype=torch.bool)
    test_mask = torch.zeros(len(y), dtype=torch.bool)

    train_mask[train_idx] = True
    test_mask[test_idx] = True

    data = data.to(device)
    data.pr = data.pr.to(device)

    model = PageRankGAT(data.x.shape[1], 128, 2).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    print("Training...")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)   
    for epoch in range(20):
        loss = train(model, data, optimizer, criterion, train_mask)
        acc, f1, auc = evaluate(model, data, test_mask)

        print(f"Epoch {epoch} | Loss {loss:.4f} | Acc {acc:.4f} | F1 {f1:.4f} | AUC {auc:.4f}")


if __name__ == "__main__":
    main()

Loading data...
Extracting features...


100%|██████████| 157/157 [14:49<00:00,  5.67s/it]


Building graph...
Computing PageRank...
Preparing data...
Training...
Epoch 0 | Loss 0.8112 | Acc 0.7750 | F1 0.7863 | AUC 0.8552
Epoch 1 | Loss 0.6325 | Acc 0.7880 | F1 0.7992 | AUC 0.8778
Epoch 2 | Loss 0.6409 | Acc 0.7980 | F1 0.8073 | AUC 0.8871
Epoch 3 | Loss 0.6084 | Acc 0.8110 | F1 0.8160 | AUC 0.8976
Epoch 4 | Loss 0.5272 | Acc 0.8100 | F1 0.8137 | AUC 0.9039
Epoch 5 | Loss 0.4857 | Acc 0.8170 | F1 0.8190 | AUC 0.9011
Epoch 6 | Loss 0.4732 | Acc 0.8240 | F1 0.8254 | AUC 0.8982
Epoch 7 | Loss 0.4596 | Acc 0.8260 | F1 0.8267 | AUC 0.9005
Epoch 8 | Loss 0.4352 | Acc 0.8220 | F1 0.8241 | AUC 0.9061
Epoch 9 | Loss 0.4170 | Acc 0.8300 | F1 0.8313 | AUC 0.9133
Epoch 10 | Loss 0.3859 | Acc 0.8470 | F1 0.8490 | AUC 0.9204
Epoch 11 | Loss 0.3640 | Acc 0.8530 | F1 0.8549 | AUC 0.9249
Epoch 12 | Loss 0.3529 | Acc 0.8520 | F1 0.8546 | AUC 0.9279
Epoch 13 | Loss 0.3526 | Acc 0.8560 | F1 0.8591 | AUC 0.9310
Epoch 14 | Loss 0.3472 | Acc 0.8590 | F1 0.8605 | AUC 0.9345
Epoch 15 | Loss 0.3325 | 

---

## Comparing models 

In [31]:
# =========================================
# MODEL COMPARISON: ViT vs GAT vs PR-GAT
# =========================================

import copy
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch_geometric.nn import GATConv

# =========================================
# 1. METRICS FUNCTIONS
# =========================================

def _safe_auc(y_true_np, y_prob_np):
    # AUC is undefined if only one class is present in a split
    if np.unique(y_true_np).size < 2:
        return float("nan")
    return roc_auc_score(y_true_np, y_prob_np)


def compute_metrics(logits, y_true, threshold=0.5):
    probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
    preds = (probs >= threshold).astype(np.int64)
    y_true_np = y_true.detach().cpu().numpy()

    acc = accuracy_score(y_true_np, preds)
    f1 = f1_score(y_true_np, preds, zero_division=0)
    auc = _safe_auc(y_true_np, probs)

    return acc, f1, auc


def tune_threshold_from_val(logits, y_true):
    probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
    y_true_np = y_true.detach().cpu().numpy()

    best_t = 0.5
    best_f1 = -1.0
    for t in np.linspace(0.1, 0.9, 17):
        preds = (probs >= t).astype(np.int64)
        f1 = f1_score(y_true_np, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)

    return best_t


# =========================================
# 2. MODEL 1: LINEAR BASELINE
# =========================================

class LinearClassifier(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, 2)

    def forward(self, x):
        return self.fc(x)


# =========================================
# 3. MODEL 2: GAT (NO PAGERANK)
# =========================================

class GATModel(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.gat = GATConv(in_dim, hidden_dim, heads=4)
        self.fc = nn.Linear(hidden_dim * 4, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.gat(x, edge_index)
        x = self.dropout(x)
        return self.fc(x)


# =========================================
# 4. MODEL 3: GAT + LEARNABLE PAGERANK GATE
# =========================================

class PageRankGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        hdim = hidden_dim * 4
        self.gat = GATConv(in_dim, hidden_dim, heads=4)
        self.dropout = nn.Dropout(0.3)
        self.pr_gate = nn.Sequential(
            nn.Linear(hdim + 1, hdim),
            nn.ReLU(),
            nn.Linear(hdim, hdim),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(hdim, 2)

    def forward(self, x, edge_index, pr):
        h = self.gat(x, edge_index)
        h = self.dropout(h)

        pr = pr.view(-1, 1)
        gate = self.pr_gate(torch.cat([h, pr], dim=1))

        # Residual-style modulation is more stable than direct scaling
        h = h + gate * h
        return self.fc(h)


# =========================================
# 5. TRAIN/EVAL HELPERS
# =========================================

def _make_weighted_ce(y_train):
    counts = torch.bincount(y_train, minlength=2).float()
    counts = torch.clamp(counts, min=1.0)
    inv = 1.0 / counts
    weights = inv / inv.sum() * 2.0
    return nn.CrossEntropyLoss(weight=weights.to(y_train.device))


def train_with_early_stopping(model, optimizer, scheduler, criterion, forward_fn, y, train_idx, val_idx, epochs=80, patience=12):
    best_state = copy.deepcopy(model.state_dict())
    best_val_auc = -np.inf
    no_improve = 0

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = forward_fn()
        loss = criterion(logits[train_idx], y[train_idx])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = forward_fn()[val_idx]
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].detach().cpu().numpy()
            val_true = y[val_idx].detach().cpu().numpy()
            val_auc = _safe_auc(val_true, val_probs)

        if np.isnan(val_auc):
            val_auc = -np.inf

        scheduler.step(val_auc if np.isfinite(val_auc) else 0.0)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model


def evaluate(model, forward_fn, y, val_idx, test_idx):
    model.eval()
    with torch.no_grad():
        logits = forward_fn()

    best_t = tune_threshold_from_val(logits[val_idx], y[val_idx])
    return compute_metrics(logits[test_idx], y[test_idx], threshold=best_t)


# =========================================
# 6. MAIN COMPARISON FUNCTION
# =========================================

def run_experiment(data, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    y = data.y
    device = data.x.device

    # Train/val/test split for stable model selection
    idx = np.arange(len(y))
    y_cpu = y.detach().cpu().numpy()

    train_idx, temp_idx = train_test_split(
        idx, test_size=0.3, stratify=y_cpu, random_state=seed
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=2/3, stratify=y_cpu[temp_idx], random_state=seed
    )

    train_idx = torch.tensor(train_idx, dtype=torch.long, device=device)
    val_idx = torch.tensor(val_idx, dtype=torch.long, device=device)
    test_idx = torch.tensor(test_idx, dtype=torch.long, device=device)

    results = {}

    # =====================================
    # MODEL 1: LINEAR
    # =====================================
    print("Training Linear Model...")
    model1 = LinearClassifier(data.x.shape[1]).to(device)
    opt1 = torch.optim.Adam(model1.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt1, mode="max", factor=0.5, patience=4)
    crit1 = _make_weighted_ce(y[train_idx])

    forward1 = lambda: model1(data.x)
    train_with_early_stopping(model1, opt1, sch1, crit1, forward1, y, train_idx, val_idx)
    results["Linear"] = evaluate(model1, forward1, y, val_idx, test_idx)

    # =====================================
    # MODEL 2: GAT
    # =====================================
    print("Training GAT Model...")
    model2 = GATModel(data.x.shape[1], 128).to(device)
    opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode="max", factor=0.5, patience=4)
    crit2 = _make_weighted_ce(y[train_idx])

    forward2 = lambda: model2(data.x, data.edge_index)
    train_with_early_stopping(model2, opt2, sch2, crit2, forward2, y, train_idx, val_idx)
    results["GAT"] = evaluate(model2, forward2, y, val_idx, test_idx)

    # =====================================
    # MODEL 3: PR-GAT
    # =====================================
    print("Training PageRank GAT...")
    model3 = PageRankGAT(data.x.shape[1], 128).to(device)
    opt3 = torch.optim.Adam(model3.parameters(), lr=1e-3, weight_decay=1e-4)
    sch3 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt3, mode="max", factor=0.5, patience=4)
    crit3 = _make_weighted_ce(y[train_idx])

    forward3 = lambda: model3(data.x, data.edge_index, data.pr)
    train_with_early_stopping(model3, opt3, sch3, crit3, forward3, y, train_idx, val_idx)
    results["PR-GAT"] = evaluate(model3, forward3, y, val_idx, test_idx)

    return results


# =========================================
# 7. PRINT RESULTS
# =========================================

def print_results(results):
    print("\n===== FINAL RESULTS =====")
    print(f"{'Model':<10} | {'Acc':<8} | {'F1':<8} | {'AUC':<8}")
    print("-" * 40)

    for model, (acc, f1, auc) in results.items():
        auc_str = "nan" if np.isnan(auc) else f"{auc:.4f}"
        print(f"{model:<10} | {acc:.4f} | {f1:.4f} | {auc_str}")

In [32]:
if "data" not in globals():
    print("Preparing graph data for comparison...")

    X_feat = extract_features(X, device)
    A = build_graph(X_feat)
    pr_scores = compute_pagerank(A)

    data = prepare_data(X_feat, pr_scores, A, y)

    # PR normalization before model use (log1p + z-score)
    pr_scores = np.log1p(pr_scores)
    pr_scores = (pr_scores - pr_scores.mean()) / (pr_scores.std() + 1e-8)
    data.pr = torch.tensor(pr_scores).float()

    data = data.to(device)
    data.pr = data.pr.to(device)

results = run_experiment(data, seed=42)
print_results(results)

Training Linear Model...
Training GAT Model...
Training PageRank GAT...

===== FINAL RESULTS =====
Model      | Acc      | F1       | AUC     
----------------------------------------
Linear     | 0.8530 | 0.8458 | 0.9398
GAT        | 0.8500 | 0.8518 | 0.9311
PR-GAT     | 0.8470 | 0.8574 | 0.9401
